In [139]:
import csv
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings


In [38]:
path_of_data = "../data/AnuraSET/raw_data/"
path_of_annotations = "../data/AnuraSET/strong_labels/"
columns_of_final_annotations = ["audiofilename", "start", "end", "label:species"]

columns_of_txt_annotations = ["start", "end", "label:species"]
annotations_extension = ".txt"
sound_extension = ".wav"

In [82]:
list_of_annotations = []
list_of_sounds = []
for folders in os.listdir(path_of_annotations):
    print(path_of_annotations + folders)
    list_of_annotations.append([path_of_annotations + folders + "/" + files for files in os.listdir(path_of_annotations+ folders + "/") if files.endswith(annotations_extension)])
    list_of_sounds.append([path_of_data + folders + "/" + files for files in os.listdir(path_of_data+ folders) if files.endswith(sound_extension)])
list_of_annotations = [item for sublist in list_of_annotations for item in sublist]
list_of_sounds = [item for sublist in list_of_sounds for item in sublist]
    

../data/AnuraSET/strong_labels/INCT17
../data/AnuraSET/strong_labels/INCT20955
../data/AnuraSET/strong_labels/INCT4
../data/AnuraSET/strong_labels/INCT41


In [141]:
df = pd.DataFrame()
for annotation in list_of_annotations:
    equivalent_sound = annotation.replace("strong_labels", "raw_data").replace(annotations_extension, sound_extension)
    if os.path.isfile(equivalent_sound):
        try:
            curr_df = pd.read_csv(annotation, sep="\t", names = columns_of_txt_annotations, lineterminator='\n')
            curr_df.insert(0, "audiofilename", equivalent_sound.split("/",3)[-1])
            
            df = pd.concat([df, curr_df])
        except pd.errors.EmptyDataError as E:
            print(annotation + " is empty")
        except pd.errors.ParserError as PE:
            print(annotation + "has extra \t at ")

df["label:species"] = df["label:species"].apply(lambda x: x.split("_")[0])

In [142]:
df

,audiofilename,start,end,label:species
0,raw_data/INCT17/INCT17_20191113_040000.wav,0.000000,59.988753,PHYSAU
1,raw_data/INCT17/INCT17_20191113_040000.wav,9.960731,10.344517,LEPFUS
2,raw_data/INCT17/INCT17_20191113_040000.wav,13.113683,13.302624,PITAZU
3,raw_data/INCT17/INCT17_20191113_040000.wav,15.422305,15.564011,PITAZU
4,raw_data/INCT17/INCT17_20191113_040000.wav,17.140487,17.229053,PITAZU
...,...,...,...,...
2,raw_data/INCT41/INCT41_20210115_050000.wav,19.130272,20.405623,BOAALB
3,raw_data/INCT41/INCT41_20210115_050000.wav,26.215557,28.955201,BOAALB
4,raw_data/INCT41/INCT41_20210115_050000.wav,32.875726,36.040487,BOAALB
5,raw_data/INCT41/INCT41_20210115_050000.wav,38.307778,46.337769,BOAALB


In [151]:
df.to_csv('/'.join(path_of_data.split("/")[:-2]) + "/annotations.csv", index=None)